In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

experiments = [
    {'name': 'TwoBandit', 'experiment': 'exp1', 'num_options': 2, 'held-out': False},
    {'name': 'TwoBandit', 'experiment': 'exp2', 'num_options': 2, 'held-out': False},
    {'name': 'DriftingBandit', 'experiment': 'exp0', 'num_options': 4, 'held-out': False},
    {'name': 'HorizonSomer', 'experiment': 'exp0', 'num_options': 2, 'held-out': False},
    {'name': 'HorizonWaltz', 'experiment': 'exp0', 'num_options': 2, 'held-out': False},
    {'name': 'HorizonSade', 'experiment': 'exp0', 'num_options': 2, 'held-out': True},
    {'name': 'HorizonFeng', 'experiment': 'exp0', 'num_options': 2, 'held-out': True},
    {'name': 'ChangingBandit', 'experiment': 'exp0', 'num_options': 2, 'held-out': True},
    {'name': 'MaggiesFarm', 'experiment': 'exp0', 'num_options': 3, 'held-out': True},
]

experiment_labels = {
    "TwoBandit_exp1": "TwoBandit - Exp.1",
    "TwoBandit_exp2": "TwoBandit - Exp.2",
    "DriftingBandit": "Drifting Bandit",
    "HorizonSomer": "Horizon Somer",
    "HorizonWaltz": "Horizon Waltz",
    "HorizonSade": "Horizon Sade",
    "HorizonFeng": "Horizon Feng",
    "ChangingBandit": "Changing Bandit",
    "MaggiesFarm": "Maggie's Farm",
    "Mean": "Mean",
}

MODEL_NAMES = {
    "LLMPredict/Base/": "Qwen3-Coder-Next - Base",
    "LLMPredict/FineTuned/": "Qwen3-Coder-Next - Fine-Tuned",
    "RescorlaWagnerResults/": "Rescorla-Wagner",
    "EvolvedCogModel/Base/": "OpenEvolve Model - Base",
    "EvolvedCogModel/FineTuned/": "OpenEvolve Model - Fine-Tuned",
}

data_paths = ['LLMPredict/Base/', 'LLMPredict/FineTuned/', 'RescorlaWagnerResults/', 'EvolvedCogModel/FineTuned/', 'EvolvedCogModel/Base/']

repo_url = (
    "https://huggingface.co/datasets/"
    "CasparFermin/llm-guided-modelling-results/resolve/main"
)

In [ ]:
# ----------- LOAD SUMMARY FILES -----------
all_rows = []

for dp in data_paths:

    df = pd.read_csv(f"{repo_url}/{dp}Results_summary.csv")

    # Create IDs consistent with the experiment list
    df["experiment_id"] = np.where(
        df["name"].eq("TwoBandit"),
        df["name"] + "_" + df["experiment"].astype(str),
        df["name"]
    )

    df["model_path"] = dp
    df["model"] = MODEL_NAMES[dp]

    all_rows.append(df)


summary = pd.concat(all_rows, ignore_index=True)

# ----------- EXPERIMENT ORDER -----------
experiments_order = []

for exp in experiments:

    if exp["name"] == "TwoBandit":
        exp_id = f"{exp['name']}_{exp['experiment']}"
    else:
        exp_id = exp["name"]

    experiments_order.append(exp_id)

# ----------- RESCORLA-WAGNER BASELINE -----------

baseline = (
    summary[
        summary["model"] == "Rescorla-Wagner"
    ]
    .set_index("experiment_id")["test_nll"]
)

# ----------- UNIFORM RANDOM BASELINE -----------
# Random-choice NLL = log(number of choices)
# Converted to the same ΔNLL scale:
# random NLL - RW NLL

num_choices = {
    exp_id: 2
    for exp_id in experiments_order
}

num_choices["MaggiesFarm"] = 3
num_choices["DriftingBandit"] = 4


random_delta = pd.Series({
    exp_id: np.log(num_choices[exp_id]) - baseline.loc[exp_id]
    for exp_id in experiments_order
})

# Equal-weight mean across experiments
random_delta = pd.concat([
    random_delta,
    pd.Series({"Mean": random_delta.mean()})
])

# ----------- DELTA NLL -----------
# model NLL - RW NLL
# Negative = better than RW
# Positive = worse than RW

summary["delta_nll"] = summary.apply(
    lambda r:
        r["test_nll"] - baseline.loc[r["experiment_id"]],
    axis=1
)


def get(model):

    return (
        summary[
            summary["model"] == model
        ]
        .set_index("experiment_id")["delta_nll"]
        .reindex(experiments_order)
    )

llm_base = get("Qwen3-Coder-Next - Base")
llm_ft   = get("Qwen3-Coder-Next - Fine-Tuned")
ev_base  = get("OpenEvolve Model - Base")
ev_ft    = get("OpenEvolve Model - Fine-Tuned")

# ----------- EXPERIMENT-WEIGHTED MEAN -----------

llm_base = pd.concat([
    llm_base,
    pd.Series(
        {"Mean": llm_base.mean()}
    )
])

llm_ft = pd.concat([
    llm_ft,
    pd.Series(
        {"Mean": llm_ft.mean()}
    )
])

ev_base = pd.concat([
    ev_base,
    pd.Series(
        {"Mean": ev_base.mean()}
    )
])

ev_ft = pd.concat([
    ev_ft,
    pd.Series(
        {"Mean": ev_ft.mean()}
    )
])


experiments_order = experiments_order + ["Mean"]

x = np.arange(len(experiments_order))

# Separator before Mean
separator_x = len(experiments_order) - 1.5


# ----------- STYLE -----------

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": "Times New Roman",
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 14,
})


fig, ax = plt.subplots(
    figsize=(10, 7)
)


color_map = {
    "Qwen3-Coder-Next - Base": "#4C78A8",
    "Qwen3-Coder-Next - Fine-Tuned": "#1F3A93",
    "OpenEvolve Model - Base": "#F58518",
    "OpenEvolve Model - Fine-Tuned": "#C75B12",
}

# ----------- LEGEND -----------
legend_handles = [

    Line2D(
        [0], [0],
        color="black",
        linewidth=2,
        label="Rescorla-Wagner"
    ),

    Line2D(
        [0], [0],
        marker="o",
        markerfacecolor=color_map["Qwen3-Coder-Next - Base"],
        markeredgecolor="white",
        markersize=10,
        linestyle="",
        label="Qwen3-Coder-Next - Base"
    ),

    Line2D(
        [0], [0],
        marker="s",
        markerfacecolor=color_map["Qwen3-Coder-Next - Fine-Tuned"],
        markeredgecolor="white",
        markersize=10,
        linestyle="",
        label="Qwen3-Coder-Next - Fine-Tuned"
    ),

    Line2D(
        [0], [0],
        marker="o",
        markerfacecolor=color_map["OpenEvolve Model - Base"],
        markeredgecolor="white",
        markersize=10,
        linestyle="",
        label="OpenEvolve Model - Base"
    ),

    Line2D(
        [0], [0],
        marker="s",
        markerfacecolor=color_map["OpenEvolve Model - Fine-Tuned"],
        markeredgecolor="white",
        markersize=10,
        linestyle="",
        label="OpenEvolve Model - Fine-Tuned"
    ),
    Line2D(
        [0], [0],
        color="gray",
        linestyle="--",
        linewidth=2,
        label="Uniform-Choice Baseline"
    ),
]

offset = 0.15

# ----------- LLM DUMBBELLS -----------
for i in range(len(experiments_order)):

    ax.plot(
        [x[i] - offset, x[i] - offset],
        [llm_base.iloc[i], llm_ft.iloc[i]],
        color="#A6C8FF",
        linewidth=4,
        alpha=0.8
    )


ax.scatter(
    x - offset,
    llm_base.values,
    s=160,
    c=color_map["Qwen3-Coder-Next - Base"],
    marker="o",
    zorder=3
)

ax.scatter(
    x - offset,
    llm_ft.values,
    s=160,
    c=color_map["Qwen3-Coder-Next - Fine-Tuned"],
    marker="s",
    zorder=3
)

# ----------- OPENEVOLVE DUMBBELLS -----------
for i in range(len(experiments_order)):

    ax.plot(
        [x[i] + offset, x[i] + offset],
        [ev_base.iloc[i], ev_ft.iloc[i]],
        color="#FFD0A1",
        linewidth=4,
        alpha=0.8
    )


ax.scatter(
    x + offset,
    ev_base.values,
    s=160,
    c=color_map["OpenEvolve Model - Base"],
    marker="o",
    zorder=3
)

ax.scatter(
    x + offset,
    ev_ft.values,
    s=160,
    c=color_map["OpenEvolve Model - Fine-Tuned"],
    marker="s",
    zorder=3
)

# ----------- UNIFORM RANDOM REFERENCE — MEAN ONLY -----------
random_width = 0.28

mean_idx = len(experiments_order) - 1
random_mean = random_delta.loc["Mean"]

ax.plot(
    [x[mean_idx] - random_width, x[mean_idx] + random_width],
    [random_mean, random_mean],
    color="gray",
    linestyle="--",
    linewidth=2,
    alpha=0.9,
    zorder=2
)

# ----------- SEPARATOR BEFORE MEAN -----------
ax.axvline(
    separator_x,
    color="gray",
    linestyle="-",
    linewidth=2,
    alpha=0.7
)

# ----------- INCLUDED / HELD-OUT GROUP LABELS -----------
first_held_out_idx = next(
    i for i, exp in enumerate(experiments)
    if exp["held-out"]
)

# Vertical separator
held_out_separator_x = first_held_out_idx - 0.5

ax.axvline(
    held_out_separator_x,
    color="gray",
    linestyle="-",
    linewidth=1,
    alpha=0.7
)

# Centers of the two groups
included_center = (0 + first_held_out_idx - 1) / 2
held_out_center = (
    first_held_out_idx + (len(experiments) - 1)
) / 2

# Labels just above the axes
ax.text(
    included_center,
    1.01,
    "Included",
    transform=ax.get_xaxis_transform(),
    ha="center",
    va="bottom",
    fontsize=16
)

ax.text(
    held_out_center,
    1.01,
    "Held-out",
    transform=ax.get_xaxis_transform(),
    ha="center",
    va="bottom",
    fontsize=16
)

# ----------- FORMATTING -----------
# RW baseline
ax.axhline(
    0,
    color="black",
    linewidth=1
)

ax.set_xticks(x)

ax.set_xticklabels(
    [experiment_labels[exp] for exp in experiments_order],
    rotation=30,
    ha="right"
)

ax.set_ylabel("Δ Test NLL")

ax.legend(
    handles=legend_handles,
    loc="best"
)

plt.tight_layout()
plt.savefig(
    "../Images/ModelPerformanceDM.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()